# TRM Compiler — Switch to Python 3.11 for Real LLVM

**Problem:** compiler_gym requires gym which only works on Python <=3.11

**Solution:** Install Python 3.11 and switch to it

⚠️ **WARNING:** This requires runtime restart after switching!

In [ ]:
# @title Step 1: Check current Python version
import sys
print(f"Current Python: {sys.version}")

In [ ]:
# @title Step 2: Install Python 3.11
# Install Python 3.11 and its venv module

!apt-get update -qq
!apt-get install -y -qq python3.11 python3.11-venv python3.11-dev

print("Python 3.11 installed!")

In [ ]:
# @title Step 3: Switch to Python 3.11

# Set Python 3.11 as default
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!sudo update-alternatives --set python3 /usr/bin/python3.11

# Verify
!python --version

In [ ]:
# @title Step 4: CRITICAL - Restart Runtime
# @markdown ⚠️ You MUST restart the runtime now!
# @markdown Go to **Runtime → Restart runtime** then run the cell below

import os
print("⚠️ PLEASE RESTART RUNTIME NOW!")
print("Go to Runtime → Restart runtime")
print("Then run the next cell to verify Python 3.11 is active")

# Kill current session to force restart
os.kill(os.getpid(), 9)

In [ ]:
# @title Step 5: Verify Python 3.11 (run AFTER restart)

import sys
print(f"Python version: {sys.version}")

if sys.version_info >= (3, 12):
    print("❌ Still on Python 3.12+ - the switch didn't work!")
    print("Try a different approach: use Docker or synthetic mode")
else:
    print("✅ Now on Python 3.11 or lower!")

In [ ]:
# @title Step 6: Install dependencies with Python 3.11
# @markdown Now we can install packages that work with Python 3.11

# Upgrade pip
!python -m pip install --upgrade pip -q

# Install numpy (compatible version)
!python -m pip install numpy -q

# Install torch
!python -m pip install torch --index-url https://download.pytorch.org/whl/cpu -q

# Install compiler_gym (should work now!)
!python -m pip install compiler_gym -q

print("✅ Dependencies installed!")

In [ ]:
# @title Step 7: Verify compiler_gym works

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["COMPILER_GYM_HOME"] = "/content/compiler_gym"

try:
    import compiler_gym
    print(f"✅ compiler_gym {compiler_gym.__version__} works!")
    
    # Quick test
    env = compiler_gym.make("llvm-v0", benchmark="cbench-v1/qsort")
    obs = env.reset()
    print(f"✅ Initial instruction count: {obs['IrInstructionCount']}")
    env.close()
    COMPILER_GYM_OK = True
except Exception as e:
    print(f"❌ compiler_gym error: {e}")
    COMPILER_GYM_OK = False

In [ ]:
# @title Step 8: Install and run TRM

# Install project
!pip uninstall -y trm-experiments >/dev/null 2>&1 || true
!pip install -e . -q

# Generate traces and train
from pathlib import Path
from torch.utils.data import DataLoader
import torch

from trm_compiler.data import CompilerTraceDataset, generate_compiler_traces
from trm_compiler.model import TinyPassOrderingRefiner, rollout_pass_optimizer
from trm_compiler.training import train_one_epoch
from trm_compiler.env_wrapper import make_compiler_env
from trm_compiler.baselines import random_search, greedy_search

# Generate traces (using REAL LLVM now!)
traces = generate_compiler_traces(
    benchmarks=['qsort', 'adpcm', 'sha'],
    episodes_per_benchmark=8,
    max_steps_per_episode=12,
    use_heuristic=not COMPILER_GYM_OK,  # Use real LLVM if available
    seed=42,
    strategy='mixed',
)
print(f'Generated {len(traces)} traces')
print(f'Mode: {"SYNTHETIC" if not COMPILER_GYM_OK else "REAL LLVM (compiler_gym)"}')

In [ ]:
# @title Step 9: Train Model

dataset = CompilerTraceDataset(traces)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0, drop_last=False)
model = TinyPassOrderingRefiner()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    losses = train_one_epoch(model, loader, optimizer, 'cpu')
    print(f'epoch={epoch+1}', {k: round(v, 4) for k, v in losses.items()})

In [ ]:
# @title Step 10: Evaluate

env = make_compiler_env('qsort', use_compilergym=COMPILER_GYM_OK)
obs, _ = env.reset()
trace = rollout_pass_optimizer(model, obs, max_steps=8, temperature=1.0, device='cpu')
print('TRM rollout steps:', len(trace))

print('Random baseline:', random_search('qsort', max_steps=12, num_trials=10, seed=42))
print('Greedy baseline:', greedy_search('qsort', max_steps=12, seed=42))